# Train the hieroglyph classifier (Colab, GPU)

Run this notebook on Colab with a GPU runtime
(Runtime -> Change runtime type -> GPU).

Two things need to get onto this Colab session before training:
1. **Code** — cloned from GitHub (this project's repo).
2. **Data** — uploaded directly from your local machine (the original
   Kaggle `archive.zip` works as-is — see step 3; Colab's storage is
   ephemeral, so this upload happens fresh each session).

## 1. Clone the repo and install it

In [ ]:
!git clone https://github.com/DelfinEryilmaz/hieroglyph-translator.git
%cd hieroglyph-translator
!pip install -e . -q

## 2. Check what torch/torchvision Colab already has

Colab comes with its own preinstalled, GPU-matched torch — we deliberately
do **not** `pip install -r requirements.txt` here, since that would pull in
our local machine's CPU-only pins and silently disable GPU training. We
just confirm a GPU is actually available.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — check Runtime > Change runtime type > GPU")

## 3. Upload the dataset

You can upload the **original `archive.zip` you downloaded from Kaggle**
directly — no need to re-zip anything, since we never modified the raw
images. Just pick that file in the upload dialog below.

(If you'd rather zip your local `data/raw/` folder instead, that also
works — the next cell handles either case automatically.)

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick archive.zip (from Kaggle) or your own data_raw.zip
uploaded_filename = next(iter(uploaded))  # don't assume a specific name

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATA_ROOT = Path("data/raw")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(uploaded_filename) as zf:
    zf.extractall(DATA_ROOT)


def class_folders(root):
    return [p for p in root.iterdir() if p.is_dir()]


top_level = class_folders(DATA_ROOT)

# Some zips (e.g. if it was re-zipped after Windows extracted it into a
# folder named after the zip file) wrap all 171 class folders inside one
# extra folder. Detect and flatten that automatically, so it doesn't matter
# which exact zip you uploaded.
if len(top_level) == 1 and len(class_folders(top_level[0])) > 1:
    wrapper = top_level[0]
    for item in wrapper.iterdir():
        shutil.move(str(item), str(DATA_ROOT / item.name))
    wrapper.rmdir()

class_dirs = class_folders(DATA_ROOT)
print(f"{len(class_dirs)} class folders extracted")
assert len(class_dirs) == 171, "Expected 171 classes — check the zip contents"

## 4. Build DataLoaders (reusing Phase 3's code)

In [ ]:
from hieroglyph.data.dataset import build_dataloaders

train_loader, val_loader, test_loader, class_to_idx = build_dataloaders(DATA_ROOT, batch_size=32)
print(f"classes: {len(class_to_idx)}")
print(f"batches -- train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

## 5. Class weights

Phase 2 found the class distribution is heavily skewed (median 5 images,
max 448). Without correcting for this, the model could get decent-looking
overall accuracy just by being good at the handful of huge classes while
ignoring rare ones. Class-weighted loss counteracts this: each class's loss
contribution is scaled inversely to how common it is, so mistakes on rare
classes count for more during training.

In [ ]:
import torch
from collections import Counter

train_labels = [label for _, label in train_loader.dataset.samples]
class_counts = Counter(class_to_idx[c] for c in train_labels)

# weight[i] = 1 / count[i], normalized so weights average to 1 — a class with
# half as many examples gets twice the loss weight per example.
num_classes = len(class_to_idx)
counts_tensor = torch.tensor([class_counts.get(i, 1) for i in range(num_classes)], dtype=torch.float)
class_weights = (1.0 / counts_tensor)
class_weights = class_weights * (num_classes / class_weights.sum())

print("min/max class weight:", class_weights.min().item(), class_weights.max().item())

## 6. Model, loss, optimizer

In [ ]:
from hieroglyph.models.classifier import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes=num_classes, pretrained=True).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
# Small LR since we're fine-tuning (large weight updates would wreck the
# pretrained features we're relying on), not training from scratch.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

## 7. Training loop

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None):
    """One pass over `loader`. Pass optimizer=None for eval (no weight updates)."""
    is_training = optimizer is not None
    model.train(is_training)

    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        with torch.set_grad_enabled(is_training):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

In [ ]:
from pathlib import Path
from hieroglyph.models.classifier import save_checkpoint

NUM_EPOCHS = 20
best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
checkpoint_path = Path("models/best_model.pt")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc = run_epoch(val_loader, model, criterion, optimizer=None)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, class_to_idx, checkpoint_path)
        marker = "  <- saved (best so far)"

    print(
        f"epoch {epoch:2d}/{NUM_EPOCHS}  "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f}  "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}{marker}"
    )

## 8. Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"best val accuracy: {best_val_acc:.3f}")

## 9. Download the trained checkpoint

This is the file that goes into your local `models/` folder for the
evaluation notebook (Phase 6) and the inference pipeline (Phase 10).

In [ ]:
from google.colab import files

files.download(str(checkpoint_path))